# AnimeLoom â€” RunPod Text-to-Anime

Generate studio-quality anime video from text on RunPod GPUs.

| Stage | What | Purpose |
|-------|------|---------|
| **Story Decomposer** | Gemini Flash (free) or rule-based | Text â†’ shot script |
| **SDXL + LoRA** | Keyframe generation | Character identity via trained LoRA |
| **AnimateDiff T2V** | Text-to-video + IP-Adapter | Real motion with character conditioning |
| **Motion Trim** | Remove static tails | Cleaner clip endings |
| **RIFE** | Temporal upscale 8â†’24fps | Smooth animation |
| **Real-ESRGAN** | Spatial sharpening | Crisp anime lines |
| **Face Restore** | Anime face enhancement | Sharper faces |
| **Color Grade** | Anime palette | Vibrant studio colors |
| **Assembly** | Cross-dissolve stitch | Smooth scene transitions |

In [ ]:
#@title 1. Setup Environment
#@markdown Run this cell once after pod starts. Installs pinned dependencies.

import subprocess, sys, os
from pathlib import Path

# Clone repo if needed
os.chdir("/workspace")
if not Path("/workspace/AnimeLoom").exists():
    subprocess.run(["git", "clone", "https://github.com/JoelJohnsonThomas/AnimeLoom.git"], check=True)
os.chdir("/workspace/AnimeLoom")
sys.path.insert(0, "/workspace/AnimeLoom")

# Memory optimization â€” reduce fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Upgrade torch to 2.5+ (required for diffusers 0.36 Wan pipeline â€” enable_gqa support)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu124",
], check=False)

# Pinned deps â€” compatible with torch 2.5.1+cu124
# diffusers 0.36.0 needed for Wan2.1 I2V pipeline support
deps = [
    "diffusers==0.36.0", "transformers==4.44.2", "accelerate==0.33.0", "numpy<2.0",
    "safetensors", "peft", "sentencepiece", "protobuf", "ftfy",
    "opencv-python-headless", "pillow", "scikit-image", "scikit-learn",
    "controlnet-aux", "einops", "omegaconf",
    "realesrgan", "basicsr==1.4.2", "gfpgan", "facexlib",
    "google-genai", "google-generativeai",
    "huggingface_hub",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + deps)

# Fix total_mem â†’ total_memory for torch 2.5
wan_path = Path("agents/animator/wan_wrapper.py")
if wan_path.exists():
    code = wan_path.read_text()
    if ".total_mem" in code and ".total_memory" not in code:
        wan_path.write_text(code.replace(".total_mem", ".total_memory"))
        print("Fixed total_mem â†’ total_memory")

# Set warehouse
os.environ["AI_CACHE_ROOT"] = "/workspace/warehouse"
for d in ["models", "lora", "outputs", "checkpoints", "references"]:
    Path(f"/workspace/warehouse/{d}").mkdir(parents=True, exist_ok=True)

# torchvision compatibility shim for basicsr
import torchvision, types
if not hasattr(torchvision.transforms, 'functional_tensor'):
    import torchvision.transforms.functional as F
    m = types.ModuleType('torchvision.transforms.functional_tensor')
    for a in ['rgb_to_grayscale', 'normalize', 'resize', 'pad']:
        if hasattr(F, a): setattr(m, a, getattr(F, a))
    sys.modules['torchvision.transforms.functional_tensor'] = m

import torch
print(f"torch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB)")
print(f"Warehouse: {os.environ['AI_CACHE_ROOT']}")
print(f"CUDA alloc config: {os.environ.get('PYTORCH_CUDA_ALLOC_CONF', 'default')}")

import diffusers
print(f"diffusers {diffusers.__version__}")
print("Setup complete!")

In [ ]:
#@title 2. Download Character LoRA from HuggingFace
#@markdown Downloads pre-trained character LoRA. Skip if you'll train your own.

CHARACTER_REPO = "joelthomas77/animeloom-sakura-haruno"  #@param {type:"string"}
CHARACTER_NAME = "sakura_haruno"  #@param {type:"string"}

from huggingface_hub import snapshot_download
import shutil
from pathlib import Path

WAREHOUSE = Path("/workspace/warehouse")
hf_dir = WAREHOUSE / "lora" / "_hf_download"

print(f"Downloading {CHARACTER_REPO}...")
snapshot_download(CHARACTER_REPO, local_dir=str(hf_dir))

# Place SDXL and SD 1.5 LoRAs in correct directories
for src, suffix in [("sdxl", ""), ("sd15", "_sd15")]:
    src_dir = hf_dir / src
    if src_dir.exists():
        dst_dir = WAREHOUSE / "lora" / f"{CHARACTER_NAME}{suffix}"
        dst_dir.mkdir(parents=True, exist_ok=True)
        for f in src_dir.iterdir():
            shutil.copy2(f, dst_dir / f.name)
        print(f"  {src} â†’ {dst_dir} ({sum(1 for _ in dst_dir.iterdir())} files)")

# Verify
sdxl_ok = (WAREHOUSE / "lora" / CHARACTER_NAME / "adapter_model.safetensors").exists()
sd15_ok = (WAREHOUSE / "lora" / f"{CHARACTER_NAME}_sd15" / "adapter_model.safetensors").exists()
print(f"\nSDXL LoRA: {'OK' if sdxl_ok else 'MISSING'}")
print(f"SD 1.5 LoRA (AnimateDiff): {'OK' if sd15_ok else 'MISSING'}")

In [ ]:
#@title 3. Text-to-Anime Generation (Wan2.1 I2V)
#@markdown **Two-stage LLM pipeline:** Gemini plans the story → Claude refines each shot cinematically.

#@markdown ---
#@markdown **Your Story**
STORY_TEXT = "Sakura Haruno walks through a cherry blossom forest at sunset. Petals fall around her as she stops at a wooden bridge and looks at the river below. The wind gently moves her pink hair."  #@param {type:"string"}

#@markdown ---
#@markdown **Character** (must match LoRA from Step 2, or leave empty)
CHARACTER_NAME = "sakura_haruno"  #@param {type:"string"}

#@markdown ---
#@markdown **API Keys**
GEMINI_API_KEY = ""  #@param {type:"string"}
#@markdown Get free Gemini key → aistudio.google.com/apikey (1500 req/day free)
ANTHROPIC_API_KEY = ""  #@param {type:"string"}
#@markdown Get Claude key → console.anthropic.com (pay-per-use, ~$0.003 per story)

#@markdown ---
#@markdown **Video Settings**
NUM_FRAMES = 33  #@param {type:"slider", min:17, max:81, step:8}
WAN_STEPS = 30  #@param {type:"slider", min:15, max:50, step:5}
WAN_GUIDANCE = 3.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
#@markdown ↑ Lower = more motion (3.0 recommended). Higher = closer to keyframe but stiffer.
FPS = 16  #@param {type:"slider", min:8, max:24, step:4}
TARGET_FPS = 24  #@param {type:"slider", min:8, max:30, step:4}

#@markdown ---
#@markdown **Post-Processing**
FACE_RESTORE = True  #@param {type:"boolean"}
SPATIAL_UPSCALE = True  #@param {type:"boolean"}
COLOR_GRADE = True  #@param {type:"boolean"}
COLOR_PALETTE = "warm"  #@param ["warm", "cool", "vibrant", "muted"]

#@markdown ---
#@markdown **SDXL Keyframe Settings**
IMAGE_WIDTH = 768  #@param {type:"slider", min:512, max:1024, step:128}
IMAGE_HEIGHT = 1152  #@param {type:"slider", min:512, max:1152, step:128}
SDXL_STEPS = 25  #@param {type:"slider", min:10, max:50, step:5}
SDXL_GUIDANCE = 7.5  #@param {type:"slider", min:1.0, max:15.0, step:0.5}
LORA_SCALE = 1.0  #@param {type:"slider", min:0.5, max:2.0, step:0.1}
NEGATIVE_PROMPT = "blurry, low quality, deformed, extra fingers, worst quality, ugly, disfigured, bad anatomy, bad hands, 3d render, cgi, photorealistic, live action, multiple views, split screen"  #@param {type:"string"}

import os, gc, sys, time, torch, json, re
import numpy as np
from pathlib import Path
from PIL import Image
import cv2

WAREHOUSE = Path(os.environ["AI_CACHE_ROOT"])
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
if ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# ================================================================
# Auto-detect GPU VRAM
# ================================================================
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if VRAM_GB >= 40:
    WAN_W, WAN_H = 480, 832
    OFFLOAD_MODE = "model"
    print(f"GPU: {VRAM_GB:.0f}GB — full resolution {WAN_W}x{WAN_H}, model CPU offload (fast)")
elif VRAM_GB >= 20:
    WAN_W, WAN_H = 480, 640
    OFFLOAD_MODE = "sequential"
    print(f"GPU: {VRAM_GB:.0f}GB — reduced resolution {WAN_W}x{WAN_H}, sequential CPU offload")
else:
    WAN_W, WAN_H = 480, 480
    OFFLOAD_MODE = "sequential"
    print(f"GPU: {VRAM_GB:.0f}GB — minimal resolution {WAN_W}x{WAN_H}")

# ================================================================
# Helpers
# ================================================================

def wan_frame_to_pil(frame):
    if isinstance(frame, Image.Image):
        return frame
    if hasattr(frame, 'cpu'):
        frame = frame.cpu().numpy()
    arr = np.squeeze(np.array(frame))
    if arr.dtype in (np.float32, np.float64, np.float16):
        arr = (arr * 255).clip(0, 255).astype(np.uint8) if arr.max() <= 1.0 else arr.clip(0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def extract_scene_environment(story_text):
    lower = story_text.lower()
    env_keywords = {
        "cherry blossom forest": "lush cherry blossom forest, pink sakura petals floating in golden sunset light, ancient twisted trees, soft volumetric god rays filtering through canopy",
        "forest": "dense mystical forest, dappled sunlight through leaves, ancient trees, ethereal atmosphere",
        "beach": "pristine anime beach, crystal turquoise water, golden sand, dramatic sunset sky",
        "city": "futuristic anime cityscape, neon lights reflecting on wet streets, towering buildings",
        "school": "Japanese high school courtyard, cherry trees, warm afternoon light",
        "temple": "ancient Japanese temple, stone lanterns, moss-covered stairs, misty atmosphere",
        "mountain": "majestic mountain landscape, dramatic clouds, alpine flowers, vast sky",
        "castle": "grand fantasy castle, towering spires, magical aurora in sky",
        "village": "quaint Japanese village, traditional wooden houses, lantern-lit streets",
        "garden": "serene Japanese garden, koi pond, stone bridge, bamboo fence",
        "bridge": "traditional wooden bridge over a gentle river, cherry blossom petals on water",
        "river": "peaceful riverside, gentle flowing water, wildflowers along banks",
        "night": "moonlit landscape, star-filled sky, soft blue ambient light",
        "sunset": "golden hour landscape, warm orange and pink sky, long dramatic shadows",
        "rain": "rain-soaked streets, puddle reflections, moody grey-blue atmosphere",
    }
    matched_env = next((desc for kw, desc in env_keywords.items() if kw in lower), None)
    if not matched_env:
        matched_env = "detailed anime environment, atmospheric lighting, painterly background"
    if "sunset" in lower or "evening" in lower:
        matched_env += ", golden hour sunset lighting, warm orange ambient"
    elif "night" in lower or "moon" in lower:
        matched_env += ", moonlit atmosphere, cool blue tones"
    elif "morning" in lower or "dawn" in lower:
        matched_env += ", soft morning light, gentle mist"
    return matched_env

# Studio anime style constants
STUDIO_STYLE = (
    "studio anime, makoto shinkai style, ufotable quality, "
    "cel shading with soft gradients, detailed lineart, "
    "cinematic composition, film grain, depth of field, "
    "volumetric lighting, painterly background art, "
    "4K anime wallpaper quality"
)

STUDIO_NEGATIVE = (
    "3d render, cgi, photorealistic, live action, photograph, "
    "blurry, low quality, worst quality, deformed face, ugly, "
    "disfigured, bad anatomy, bad hands, extra fingers, "
    "static image, no motion, frozen, still frame, "
    "watermark, text, logo, chibi, super deformed, sketch, "
    "multiple views, split screen, western cartoon, disney style"
)

# ================================================================
# Phase 1: Story Decomposition (Gemini → Claude)
# ================================================================
print("=" * 60)
print("PHASE 1: Story Decomposition")
print("=" * 60)

_mode = "Two-stage (Gemini + Claude)" if (GEMINI_API_KEY and ANTHROPIC_API_KEY) else \
        ("Gemini only" if GEMINI_API_KEY else "Rule-based fallback")
print(f"  Mode: {_mode}\n")

from agents.story.decomposer import StoryDecomposer
decomposer = StoryDecomposer(
    gemini_api_key=GEMINI_API_KEY or None,
    anthropic_api_key=ANTHROPIC_API_KEY or None,
    character_name=CHARACTER_NAME or None,
)
shots_raw = decomposer.decompose_to_shots(STORY_TEXT)

# Enforce single consistent scene across all shots
scene_env = extract_scene_environment(STORY_TEXT)
camera_sequence = [
    "wide establishing shot slowly pushing in",
    "medium shot tracking alongside, gentle dolly",
    "close-up portrait shot, shallow depth of field",
    "medium-wide pullback shot, scenic composition",
    "over-the-shoulder shot, atmospheric depth",
]

# Action-rich motion cues for Wan (used when Claude refinement isn't available)
action_motion_pool = [
    "character walks slowly forward along path, arms swaying gently at sides, hair and skirt flowing with each step, cherry blossom petals drift around her",
    "character pauses mid-step and turns head to the side, one hand rises to brush hair from face, fabric ripples in breeze, soft petals float past",
    "character leans forward against wooden bridge railing, looks down at water, hair falls forward over shoulder, reflected light dances on her face",
    "character straightens up from railing, turns body slightly, reaches one hand toward a falling petal, wind catches her hair and lifts it",
    "character takes a slow step forward, tilts head up toward sky, arms relaxed at sides with gentle sway, petals swirl in golden light around her",
]

shots = []
for i, raw in enumerate(shots_raw):
    shots.append({
        "description": scene_env,
        "action": raw.get("action") or raw.get("description", ""),
        "camera": raw.get("camera") or camera_sequence[i % len(camera_sequence)],
        "mood": raw.get("mood", "cinematic, atmospheric"),
        "characters": raw.get("characters", [CHARACTER_NAME or "Character"]),
        "refined_prompt": raw.get("description", "") if _mode.startswith("Two") else "",
    })

if not shots:
    print("  No shots — using fallback")
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', STORY_TEXT.strip()) if s.strip()]
    for i, sent in enumerate(sentences):
        shots.append({
            "description": scene_env,
            "action": sent.rstrip("."),
            "camera": camera_sequence[i % len(camera_sequence)],
            "mood": "cinematic", "characters": [CHARACTER_NAME or "Character"],
            "refined_prompt": "",
        })

print(f"\nScene: {scene_env[:80]}...")
print(f"\n{len(shots)} shots:")
for i, s in enumerate(shots):
    print(f"  Shot {i+1}: [{s['camera'][:25]}] {s['action'][:55]}...")

# ================================================================
# Phase 2: SDXL + LoRA Keyframes
# ================================================================
print("\n" + "=" * 60)
print("PHASE 2: SDXL + LoRA Keyframes")
print("=" * 60)

gc.collect(); torch.cuda.empty_cache()

char_id = CHARACTER_NAME.lower().replace(" ", "_") if CHARACTER_NAME else None
lora_dir = WAREHOUSE / "lora" / char_id if char_id else None
has_lora = lora_dir and (lora_dir / "metadata.json").exists() if lora_dir else False

from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if has_lora:
    from peft import PeftModel
    meta = json.loads((lora_dir / "metadata.json").read_text())
    base_model = meta.get("base_model", "cagliostrolab/animagine-xl-3.1")
    if base_model.startswith("/") or base_model.startswith("C:"):
        base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} + LoRA for '{CHARACTER_NAME}'...")
else:
    base_model = "cagliostrolab/animagine-xl-3.1"
    print(f"Loading {base_model} (no LoRA)...")

pipe = StableDiffusionXLPipeline.from_pretrained(
    base_model, torch_dtype=torch.float16, cache_dir=str(WAREHOUSE / "models"),
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

if has_lora and (lora_dir / "adapter_model.safetensors").exists():
    pipe.unet = PeftModel.from_pretrained(pipe.unet, str(lora_dir))
    pipe.unet.eval()
    if LORA_SCALE != 1.0:
        for module in pipe.unet.modules():
            if hasattr(module, "scaling"):
                for key in module.scaling:
                    module.scaling[key] = LORA_SCALE
    print(f"  LoRA loaded (scale={LORA_SCALE})")

pipe.to("cuda")
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
try: pipe.enable_xformers_memory_efficient_attention()
except: pass

keyframes = []
char_tag = CHARACTER_NAME.replace("_", " ").title() if CHARACTER_NAME else "anime girl"

for i, shot in enumerate(shots):
    prompt = (
        f"1girl, {char_tag}, "
        f"beautiful detailed face, clear sharp eyes, expressive anime eyes, detailed hair, "
        f"{scene_env}, {shot['camera']}, "
        f"{STUDIO_STYLE}, masterpiece, best quality, absurdres"
    )
    gen = torch.Generator("cuda").manual_seed(42 + i)
    result = pipe(
        prompt=prompt, negative_prompt=NEGATIVE_PROMPT,
        height=IMAGE_HEIGHT, width=IMAGE_WIDTH,
        num_inference_steps=SDXL_STEPS, guidance_scale=SDXL_GUIDANCE, generator=gen,
    )
    keyframes.append(result.images[0])
    print(f"  Keyframe {i+1}/{len(shots)}: {shot['camera'][:50]}")

kf_dir = WAREHOUSE / "outputs" / "keyframes"
kf_dir.mkdir(parents=True, exist_ok=True)
for i, kf in enumerate(keyframes):
    kf.save(kf_dir / f"kf_{i:03d}.png")

if has_lora:
    while hasattr(pipe.unet, "base_model"):
        try: pipe.unet = pipe.unet.base_model.model
        except: break
pipe.to("cpu")
del pipe; gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
print(f"SDXL unloaded. VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

# ================================================================
# Phase 3: Wan2.1 Image-to-Video (guidance=3.0 for real motion)
# ================================================================
print("\n" + "=" * 60)
print(f"PHASE 3: Wan2.1 I2V (guidance={WAN_GUIDANCE} — {'motion-rich' if WAN_GUIDANCE <= 3.5 else 'stiff'})")
print("=" * 60)

all_clips = []
start_time = time.time()

from diffusers import WanImageToVideoPipeline
WAN_MODEL = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"
print(f"Loading {WAN_MODEL}...")

wan_pipe = WanImageToVideoPipeline.from_pretrained(
    WAN_MODEL, torch_dtype=torch.float16, cache_dir=str(WAREHOUSE / "models"),
)
if OFFLOAD_MODE == "model":
    wan_pipe.enable_model_cpu_offload()
else:
    wan_pipe.enable_sequential_cpu_offload()
wan_pipe.vae.enable_slicing()
wan_pipe.vae.enable_tiling()
print(f"Wan2.1 ready — {OFFLOAD_MODE} offload, {WAN_W}x{WAN_H}, guidance={WAN_GUIDANCE}\n")

for i, kf_image in enumerate(keyframes):
    shot = shots[i]

    # Use Claude-refined prompt if available, otherwise use action motion pool
    if shot.get("refined_prompt") and len(shot["refined_prompt"]) > 30:
        wan_prompt = (
            f"{shot['refined_prompt']}, "
            f"smooth animation, studio anime quality, film grain, cinematic"
        )
    else:
        motion = action_motion_pool[i % len(action_motion_pool)]
        wan_prompt = (
            f"{motion}, "
            f"smooth animation, studio anime quality, cinematic, film grain"
        )

    print(f"  Shot {i+1}/{len(keyframes)}:")
    print(f"    Prompt: {wan_prompt[:90]}...")
    print(f"    VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

    kf_resized = kf_image.resize((WAN_W, WAN_H), Image.LANCZOS)
    gen = torch.Generator("cpu").manual_seed(42 + i)

    result = wan_pipe(
        image=kf_resized,
        prompt=wan_prompt,
        negative_prompt=STUDIO_NEGATIVE,
        num_frames=NUM_FRAMES,
        width=WAN_W,
        height=WAN_H,
        num_inference_steps=WAN_STEPS,
        guidance_scale=WAN_GUIDANCE,
        generator=gen,
    )
    clip_frames = result.frames[0]
    processed_frames = [wan_frame_to_pil(f) for f in clip_frames]

    if len(processed_frames) >= 2:
        diff = np.mean(np.abs(
            np.array(processed_frames[0]).astype(np.float32) -
            np.array(processed_frames[-1]).astype(np.float32)
        ))
        status = "Good motion" if diff >= 5.0 else ("Some motion" if diff >= 1.0 else "WARNING: nearly static")
        print(f"    {status} (diff={diff:.1f})")

    all_clips.append(processed_frames)
    print(f"    {len(processed_frames)} frames")
    gc.collect(); torch.cuda.empty_cache()

anim_time = time.time() - start_time
print(f"\nWan2.1 done in {anim_time/60:.1f} min")
del wan_pipe; gc.collect(); torch.cuda.empty_cache()

# ================================================================
# Phase 4: Post-Processing
# ================================================================
print("\n" + "=" * 60)
print("PHASE 4: Post-Processing")
print("=" * 60)

try:
    from agents.postprocess.motion_trim import MotionTrimmer
    trimmer = MotionTrimmer()
    for idx, clip in enumerate(all_clips):
        orig = len(clip)
        all_clips[idx] = trimmer.trim_static_tail(clip)
        trimmed = orig - len(all_clips[idx])
        if trimmed > 0:
            print(f"  Clip {idx+1}: trimmed {trimmed} static frames")
except Exception as e:
    print(f"  Motion trim skipped: {e}")

if FACE_RESTORE:
    try:
        from agents.postprocess.face_restore import AnimeFaceRestorer
        restorer = AnimeFaceRestorer()
        print("Restoring anime faces...")
        for idx, clip in enumerate(all_clips):
            all_clips[idx] = restorer.restore_frames(clip)
        print("  Done")
    except Exception as e:
        print(f"  Face restore skipped: {e}")

if SPATIAL_UPSCALE:
    try:
        from basicsr.archs.rrdbnet_arch import RRDBNet
        from realesrgan import RealESRGANer
        esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=6, num_grow_ch=32, scale=4)
        esrgan = RealESRGANer(
            scale=4,
            model_path="https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth",
            model=esrgan_model, half=True,
        )
        print("Sharpening with Real-ESRGAN...")
        for idx, clip in enumerate(all_clips):
            sharpened = []
            for frame_pil in clip:
                frame_bgr = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)
                h_orig, w_orig = frame_bgr.shape[:2]
                out_bgr, _ = esrgan.enhance(frame_bgr, outscale=1)
                if out_bgr.shape[:2] != (h_orig, w_orig):
                    out_bgr = cv2.resize(out_bgr, (w_orig, h_orig), interpolation=cv2.INTER_LANCZOS4)
                sharpened.append(Image.fromarray(cv2.cvtColor(out_bgr, cv2.COLOR_BGR2RGB)))
            all_clips[idx] = sharpened
        del esrgan, esrgan_model; gc.collect(); torch.cuda.empty_cache()
        print("  Done")
    except Exception as e:
        print(f"  Real-ESRGAN skipped: {e}")

if COLOR_GRADE:
    try:
        from agents.postprocess.color_grade import AnimeColorGrader
        grader = AnimeColorGrader()
        print(f"Color grading ({COLOR_PALETTE})...")
        for idx, clip in enumerate(all_clips):
            all_clips[idx] = grader.grade_with_palette(clip, palette=COLOR_PALETTE)
        print("  Done")
    except Exception as e:
        print(f"  Color grading skipped: {e}")

# ================================================================
# Phase 5: Temporal Upscale (RIFE)
# ================================================================
if TARGET_FPS > FPS:
    print(f"\nTemporal upscale: {FPS}fps -> {TARGET_FPS}fps")
    multiplier = max(2, round(TARGET_FPS / FPS))
    rife_ok = False
    try:
        from agents.postprocess.upscaler import VideoUpscaler
        _up = VideoUpscaler(str(WAREHOUSE))
        _rife = _up._load_rife()
        if _rife: rife_ok = True; print("  Using RIFE")
    except: pass

    for idx, clip in enumerate(all_clips):
        orig = len(clip)
        if rife_ok:
            try:
                all_clips[idx] = _up._rife_interpolate_sequence(_rife, clip, multiplier)
            except:
                rife_ok = False
        if not rife_ok:
            upscaled = []
            for j in range(len(clip) - 1):
                upscaled.append(clip[j])
                a1 = np.array(clip[j]).astype(np.float32)
                a2 = np.array(clip[j+1]).astype(np.float32)
                for k in range(1, multiplier):
                    alpha = k / multiplier
                    upscaled.append(Image.fromarray(((1-alpha)*a1 + alpha*a2).astype(np.uint8)))
            upscaled.append(clip[-1])
            all_clips[idx] = upscaled
        print(f"  Clip {idx+1}: {orig} -> {len(all_clips[idx])} frames")

    if rife_ok:
        try: _up._unload_rife()
        except: pass
        gc.collect(); torch.cuda.empty_cache()
    output_fps = TARGET_FPS
else:
    output_fps = FPS

# ================================================================
# Phase 6: Assembly
# ================================================================
print(f"\nAssembling {len(all_clips)} clips...")
CROSSFADE = 12
all_frames = []
for idx, clip in enumerate(all_clips):
    arrays = [np.array(f) for f in clip]
    if idx == 0:
        all_frames.extend(arrays)
    else:
        overlap = min(CROSSFADE, len(all_frames), len(arrays))
        for j in range(overlap):
            t = (j + 1) / (overlap + 1)
            prev = all_frames[-(overlap - j)].astype(np.float32)
            nxt = arrays[j].astype(np.float32)
            all_frames[-(overlap - j)] = ((1-t)*prev + t*nxt).clip(0, 255).astype(np.uint8)
        all_frames.extend(arrays[overlap:])

duration = len(all_frames) / output_fps
print(f"Total: {len(all_frames)} frames ({duration:.1f}s at {output_fps}fps)")

output_dir = WAREHOUSE / "outputs" / "text2anime"
output_dir.mkdir(parents=True, exist_ok=True)
video_path = output_dir / f"anime_{int(time.time())}.mp4"

h, w = all_frames[0].shape[:2]
writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*"mp4v"), output_fps, (w, h))
for frame in all_frames:
    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
writer.release()

total_time = time.time() - start_time
size_mb = video_path.stat().st_size / 1e6

print(f"\n{'=' * 60}")
print(f"DONE! [{_mode}]")
print(f"{'=' * 60}")
print(f"Video:    {video_path}")
print(f"Duration: {duration:.1f}s | Res: {w}x{h} | FPS: {output_fps}")
print(f"Size:     {size_mb:.1f}MB | Time: {total_time/60:.1f} min")
print(f"Wan guidance: {WAN_GUIDANCE} ({'motion-rich' if WAN_GUIDANCE <= 3.5 else 'stable'})")

import matplotlib.pyplot as plt
cols = min(len(keyframes), 4)
rows = (len(keyframes) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axes_flat = np.array(axes).flat if hasattr(axes, 'flat') else [axes]
for i, (ax, kf) in enumerate(zip(axes_flat, keyframes)):
    ax.imshow(kf)
    ax.set_title(f"Shot {i+1}: {shots[i]['action'][:35]}...", fontsize=9)
    ax.axis("off")
for j in range(i+1, rows*cols):
    try: axes_flat[j].axis("off")
    except: pass
plt.suptitle(f"Keyframes — {_mode}", fontsize=13)
plt.tight_layout()
plt.show()

gc.collect(); torch.cuda.empty_cache()
print("VRAM freed.")

In [ ]:
#@title 2.5 Patch Story Decomposer (two-stage Gemini → Claude pipeline)
#@markdown Writes the updated decomposer.py with Claude refinement support.
#@markdown Only needed until the repo is updated — run once per pod.

import os
from pathlib import Path

decomposer_path = Path("AnimeLoom/agents/story/decomposer.py")
decomposer_path.parent.mkdir(parents=True, exist_ok=True)

decomposer_code = r'''"""
Story Decomposer — two-stage LLM pipeline (Gemini → Claude) for anime shot scripts.
"""

import os
import re
from typing import Dict, List, Optional


class StoryDecomposer:

    _GEMINI_SYSTEM = """\
You are an anime series director and story planner.

Given a story description, produce a STRUCTURED SHOT LIST.
For each shot output EXACTLY this format (no markdown, no extra text):

SCENE: <one consistent background/environment for ALL shots — do not change between shots>
CHAR: <character name>
ACTION: <what the character physically does — MUST include clear body movement like walking, reaching, turning, leaning. 2-3 sentences.>
CAMERA: <camera movement and framing: wide/medium/close-up + dolly/pan/tracking/static>
MOOD: <emotional tone + lighting keyword>

Rules:
- ALL shots share the SAME scene/environment (single continuous location)
- Each shot MUST have clear visible character movement — NOT static poses
- Good actions: walking forward, slight head turns, reaching out, looking down, leaning on railing
- Avoid: full body spins, jumping, running, extreme head rotations
- 3-5 shots total
- Output ONLY the shot list
"""

    _CLAUDE_REFINE_SYSTEM = """\
You are a sakuga animator and cinematographer for ufotable / Makoto Shinkai.

Refine a rough shot into a cinematic anime prompt for an image-to-video AI model.

CRITICAL RULES:
1. The character MUST have visible body movement — walking, arm gestures, head tilts, leaning, reaching. NEVER describe a static pose.
2. Describe both CHARACTER motion AND environmental motion (hair flowing, petals, fabric, light shifts)
3. Use anime visual language: volumetric light, depth of field, film grain, sakuga, chromatic aberration
4. Keep the same background/scene as other shots
5. Avoid: full body rotations, jumping, sudden teleportation, extreme head turns
6. Good: "walks slowly forward, arms swaying gently, hair caught by wind" / "turns head left and gazes downward, one hand reaching toward falling petals"

Output ONLY the refined prompt (2-4 sentences). No labels, no markdown.
"""

    def __init__(self, gemini_api_key=None, anthropic_api_key=None, character_name=None):
        self._gemini_key = gemini_api_key or os.getenv("GEMINI_API_KEY")
        self._anthropic_key = anthropic_api_key or os.getenv("ANTHROPIC_API_KEY")
        self._character_name = character_name

    def decompose(self, text):
        if not text.strip():
            return ""
        if self._is_already_script(text):
            return text
        if self._gemini_key and self._anthropic_key:
            result = self._decompose_two_stage(text)
            if result:
                return result
        if self._gemini_key:
            result = self._decompose_via_gemini(text)
            if result:
                return result
        return self._decompose_local(text)

    def decompose_to_shots(self, text):
        script = self.decompose(text)
        return self._parse_script(script)

    def _decompose_two_stage(self, text):
        print("  [Two-stage] Stage 1: Gemini story planning...")
        gemini_script = self._decompose_via_gemini(text)
        if not gemini_script:
            return None
        shots = self._parse_script(gemini_script)
        if not shots:
            return None
        print(f"  [Two-stage] Gemini produced {len(shots)} shots. Stage 2: Claude refinement...")
        refined_lines = []
        for i, shot in enumerate(shots):
            raw_desc = shot.get("description", "")
            action = shot.get("action", raw_desc)
            camera = shot.get("camera", "")
            mood = shot.get("mood", "")
            chars = shot.get("characters", [self._character_name or "Character"])
            refined = self._refine_shot_via_claude(
                shot_index=i + 1, total_shots=len(shots),
                action=action, camera=camera, mood=mood,
                character=chars[0] if chars else (self._character_name or "Character"),
            )
            refined_lines.append(f"SCENE: {raw_desc}")
            refined_lines.append(f"CHAR: {', '.join(chars)}")
            if camera:
                refined_lines.append(f"CAMERA: {camera}")
            refined_lines.append(refined)
            refined_lines.append("")
        print(f"  [Two-stage] Done — {len(shots)} shots refined by Claude.")
        return "\n".join(refined_lines)

    def _refine_shot_via_claude(self, shot_index, total_shots, action, camera, mood, character):
        try:
            import anthropic
            client = anthropic.Anthropic(api_key=self._anthropic_key)
            user_msg = (
                f"Shot {shot_index} of {total_shots}.\n"
                f"Character: {character}\nAction: {action}\n"
                f"Camera: {camera}\nMood: {mood}\n\n"
                f"Refine into a cinematic anime shot. The character MUST have visible body movement."
            )
            response = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=300,
                system=self._CLAUDE_REFINE_SYSTEM,
                messages=[{"role": "user", "content": user_msg}],
            )
            refined = response.content[0].text.strip()
            print(f"    Shot {shot_index}: refined ({len(refined)} chars)")
            return refined
        except ImportError:
            print("    Claude SDK not installed — using raw action text")
            return action
        except Exception as e:
            print(f"    Claude refinement failed for shot {shot_index}: {e}")
            return action

    def _decompose_via_gemini(self, text):
        result = self._try_genai_new(text)
        if result:
            return result
        return self._try_genai_legacy(text)

    def _try_genai_new(self, text):
        try:
            from google import genai
            client = genai.Client(api_key=self._gemini_key)
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"{self._GEMINI_SYSTEM}\n\nStory:\n{text}",
                config={"temperature": 0.6, "max_output_tokens": 2048},
            )
            result = response.text.strip()
            if "SCENE:" in result and "CHAR:" in result:
                print("  Gemini 2.5 Flash story planning done (new SDK)")
                return result
            return None
        except ImportError:
            return None
        except Exception as e:
            print(f"  Gemini (new SDK) failed: {e}")
            return None

    def _try_genai_legacy(self, text):
        try:
            import google.generativeai as genai
            genai.configure(api_key=self._gemini_key)
            for model_name in ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash"]:
                try:
                    model = genai.GenerativeModel(model_name)
                    response = model.generate_content(
                        [{"role": "user", "parts": [f"{self._GEMINI_SYSTEM}\n\nStory:\n{text}"]}],
                        generation_config=genai.GenerationConfig(temperature=0.6, max_output_tokens=2048),
                    )
                    result = response.text.strip()
                    if "SCENE:" in result and "CHAR:" in result:
                        print(f"  Gemini story planning done ({model_name}, legacy SDK)")
                        return result
                except Exception:
                    continue
            return None
        except ImportError:
            return None
        except Exception as e:
            print(f"  Gemini (legacy SDK) failed: {e}")
            return None

    def _decompose_local(self, text):
        sentences = self._split_sentences(text)
        characters = self._extract_characters(text)
        if self._character_name and self._character_name not in characters:
            characters.insert(0, self._character_name)
        lines = []
        for i, sentence in enumerate(sentences):
            setting = self._infer_setting(sentence)
            chars = self._find_characters_in_text(sentence, characters)
            if not chars:
                chars = [self._character_name] if self._character_name else ["Character"]
            lines.append(f"SCENE: {setting}")
            lines.append(f"CHAR: {', '.join(chars)}")
            lines.append(self._build_anime_prompt(sentence))
            lines.append("")
        return "\n".join(lines)

    def _split_sentences(self, text):
        return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.strip()) if s.strip()]

    def _extract_characters(self, text):
        words = re.findall(r'\b([A-Z][a-z]{2,})\b', text)
        stop = {"The","This","That","There","They","Then","Their","She","Her","His","Him","But","And","For","With","From","Into","Through","After","Before","When","Where","While","During","About","Each","Every","Some","Many","Scene","Shot","Camera","Petals","Wind","River","Bridge","Forest","Cherry","Blossom","Sunset","Night","Morning","Light","Dark","Rain","Snow","Fire","Water","Sky"}
        from collections import Counter
        wc = Counter(words)
        return [w for w, c in wc.items() if w not in stop and (c >= 2 or len(words) <= 5)][:5]

    def _infer_setting(self, text):
        lower = text.lower()
        for pat in [r'(?:at|in|inside|outside)\s+(?:the|a)\s+(\w+(?:\s+\w+)?)', r'(?:forest|garden|school|temple|city|street|room|house|beach|mountain|castle|village)']:
            m = re.search(pat, lower)
            if m:
                return f"{m.group(0).title()}, anime environment, atmospheric lighting"
        return "Dramatic anime scene, detailed background, atmospheric lighting"

    def _find_characters_in_text(self, text, all_characters):
        return [c for c in all_characters if c.lower() in text.lower()][:2]

    def _build_anime_prompt(self, text):
        return f"{text.strip().rstrip('.')}, anime style, high quality animation, detailed character art, vibrant colors, smooth motion"

    def _is_already_script(self, text):
        u = text.upper()
        return "SCENE:" in u and "CHAR:" in u

    def _parse_script(self, script):
        lines = script.strip().split("\n")
        shots, current = [], {"characters": [], "description": "", "action": "", "camera": "", "mood": "", "pose_ref": None}
        for line in lines:
            line = line.strip()
            if not line:
                continue
            u = line.upper()
            if u.startswith("SCENE:") or u.startswith("SHOT:"):
                if current["description"]:
                    shots.append(current)
                tl = 6 if u.startswith("SCENE:") else 5
                current = {"characters": [], "description": line[tl:].strip(), "action": "", "camera": "", "mood": "", "pose_ref": None}
            elif u.startswith("CHAR:"):
                for n in line[5:].strip().split(","):
                    n = n.strip()
                    if n and n not in current["characters"]:
                        current["characters"].append(n)
            elif u.startswith("ACTION:"): current["action"] = line[7:].strip()
            elif u.startswith("CAMERA:"): current["camera"] = line[7:].strip()
            elif u.startswith("MOOD:"): current["mood"] = line[5:].strip()
            elif u.startswith("POSE:"): current["pose_ref"] = line[5:].strip()
            else:
                current["description"] += (" " if current["description"] else "") + line
        if current["description"]:
            shots.append(current)
        return shots
'''

decomposer_path.write_text(decomposer_code)

import importlib
import agents.story.decomposer as _dsm
importlib.reload(_dsm)

print("Decomposer patched — Claude now generates motion-rich prompts")

In [ ]:
#@title 4. Play Video
#@markdown Displays the generated video inline.

from IPython.display import Video, display
from pathlib import Path
import glob

# Find latest video
output_dir = Path("/workspace/warehouse/outputs/text2anime")
videos = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if videos:
    latest = videos[0]
    print(f"Playing: {latest}")
    print(f"Size: {latest.stat().st_size / 1e6:.1f}MB")
    display(Video(str(latest), embed=True, width=512))
else:
    print("No videos found. Run Cell 3 first.")

In [ ]:
#@title 5. Download Video
#@markdown Downloads the latest video to your local machine.

from pathlib import Path
from IPython.display import FileLink

output_dir = Path("/workspace/warehouse/outputs/text2anime")
videos = sorted(output_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
if videos:
    print(f"Right-click to download: {videos[0].name}")
    display(FileLink(str(videos[0])))
else:
    print("No videos found.")

In [ ]:
#@title 6. Stop Pod (saves money!)
#@markdown Stops the pod to avoid charges. Your /workspace volume persists.

STOP_POD = False  #@param {type:"boolean"}

if STOP_POD:
    import subprocess
    subprocess.run(["runpodctl", "stop", "pod", "$RUNPOD_POD_ID"], check=False)
    print("Pod stopping... Your volume data is preserved.")
else:
    print("Toggle STOP_POD to True and re-run to stop.")